In [2]:
# Ignore warning
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import glob, os
import matplotlib.pyplot as plt
import numpy as np
import geopandas
import netCDF4
import h5py
import datetime as dt
import pyproj

# check pytorch version
import torch    
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim    

from tqdm import tqdm

from pyproj import Proj, transform
from shapely.geometry import Polygon, Point, LineString, shape
import cartopy.crs as ccrs
import torch

import time

from scipy.interpolate import griddata

# import cdsapi
import xarray as xr
from urllib.request import urlopen

from urllib.request import urlretrieve

import pickle

import scipy.io as sio

%load_ext autoreload
%autoreload 2

import dgl
from dgl.data import DGLDataset
from dgl import save_graphs, load_graphs
import torch
import os

from functions import *
from DGL_model import *

# GNN data read & Conversion into CNN data (Helheim dataset)

## Read ISSM simulation results

In [3]:
filename = "D:\\ISSM\\Helheim\\Helheim_r070.mat"
test = sio.loadmat(filename)

In [ ]:
test = sio.loadmat(filename)
rate = int(filename[-7:-4])*10

xc = test['S'][0][0][0]
yc = test['S'][0][0][1]
elements = test['S'][0][0][2]-1

smb = test['S'][0][0][3]
vx = test['S'][0][0][4]
vy = test['S'][0][0][5]
vel = test['S'][0][0][6]
surface = test['S'][0][0][7]
base = test['S'][0][0][8]
H = test['S'][0][0][9]
f = test['S'][0][0][10]

n_year, n_sample = H.shape

for t in tqdm(range(0, n_year)):
    # INPUT: x/y coordinates, melting rate, time, SMB, Vx0, Vy0, Surface0, Base0, Thickness0, Floating0
    inputs = torch.zeros([n_sample, 11])
    # OUTPUT: Vx, Vy, Vel, Surface, Thickness, Floating
    outputs = torch.zeros([n_sample, 6])
    
    ## INPUTS ================================================
    inputs[:, 0] = torch.tensor((xc[:, 0]-xc.min())/(xc.max()-xc.min())) # X coordinate
    inputs[:, 1] = torch.tensor((yc[:, 0]-yc.min())/(yc.max()-yc.min())) # Y coordinate
    inputs[:, 2] = torch.tensor(rate*0.001) # Melting rate
    inputs[:, 3] = torch.tensor(t/n_year) # Year
    inputs[:, 4] = torch.tensor(smb[t, :]) # Surface mass balance
    inputs[:, 5] = torch.tensor(vx[0, :]/5000) # Initial Vx
    inputs[:, 6] = torch.tensor(vy[0, :]/5000) # Initial Vx
    inputs[:, 7] = torch.tensor(surface[0, :]/4000) # Initial surface elevation
    inputs[:, 8] = torch.tensor(base[0, :]/4000) # Initial base elevation
    inputs[:, 9] = torch.tensor(H[0, :]/4000) # Initial ice thickness
    inputs[:, 10] = torch.tensor(f[0, :]/3000) # Initial floating part
    
    ## OUTPUTS ===============================================
    outputs[:, 0] = torch.tensor(vx[t, :]/5000) # Initial Vx
    outputs[:, 1] = torch.tensor(vy[t, :]/5000) # Initial Vx
    outputs[:, 2] = torch.tensor(vel[t, :]/5000) # Initial surface elevation
    outputs[:, 3] = torch.tensor(surface[t, :]/4000) # Initial base elevation
    outputs[:, 4] = torch.tensor(H[t, :]/4000) # Initial ice thickness
    outputs[:, 5] = torch.tensor(f[t, :]/3000) # Initial floating part

    # for i in range(0, n_sample):        
    #     inputs[i, :] = torch.tensor([(xc[i, 0]-xc.min())/(xc.max()-xc.min()), (yc[i, 0]-yc.min())/(yc.max()-yc.min()), rate*0.001, t/n_year, smb[t,i],
    #                                  vx[0, i]/5000, vy[0, i]/5000, surface[0, i]/4000, base[0,i]/4000, H[0,i]/4000, f[0,i]/3000
    #                                 ])
    #     outputs[i, :] = torch.tensor([vx[t, i]/5000, vy[t, i]/5000, vel[t,i]/5000, surface[t, i]/4000, H[t,i]/4000, f[t,i]/3000])

## Valid grid generation

In [23]:
# GRID FILTERING ==========================================================================

xext = [xc.min(), xc.max()]
yext = [yc.min(), yc.max()]
res = 1000

gridx, gridy = np.meshgrid(np.arange(xext[0], xext[1], res), np.arange(yext[0], yext[1], res))
coord = (xc[:, 0], yc[:, 0]) #np.array([xc[:, 0], yc[:, 0]]).transpose()

# Read boundary matrix
test = sio.loadmat("D:\\ISSM\\Helheim\\Helheim_boundary.mat")
boundary = test['boundary']
bind = np.where(boundary == 1)[0]

df = pd.DataFrame()
points = []

for i, ind in enumerate(bind):
    points.append((xc[ind][0], yc[ind][0]))
    df.loc[i, "X"] = xc[ind][0]
    df.loc[i, "Y"] = yc[ind][0]

xn, yn = sort_xy(df["X"].values, df["Y"].values)
df1 = pd.DataFrame({'X': xn, 'Y': yn})   

geometry = [Point(xy) for xy in zip(df1.X, df1.Y)]
gdf = geopandas.GeoDataFrame(df1, geometry=geometry)

# Polygon of Helheim Glacier
gdf['shape_id'] = 0
polygon = gdf.groupby('shape_id')['geometry'].apply(lambda x: Polygon(x.tolist())).reset_index()

grid = []
grid_i = []
grid_j = []

for i in range(0, gridx.shape[0]):
    for j in range(0, gridx.shape[1]):
        grid.append(Point(gridx[i, j], gridy[i, j]))
        grid_i.append(i)
        grid_j.append(j)

gridf = geopandas.GeoDataFrame(pd.DataFrame({'ind_i': grid_i, 'ind_j': grid_j}), geometry=grid)

pointInPoly = geopandas.sjoin(gridf, polygon, op='within') 

grid_valid = gridx.copy() * np.nan
grid_valid[pointInPoly['ind_i'].values, pointInPoly['ind_j'].values] = 1
print("Polygon prepared!")
# =========================================================================================

496
Polygon prepared!


## Convert GNN data to CNN data

In [11]:
filename = f'D:\\ISSM\\Helheim\\Helheim_r100_030.mat'
    
rate = int(filename[-11:-8])

test = sio.loadmat(filename)

xc = test['S'][0][0][0]
yc = test['S'][0][0][1]
elements = test['S'][0][0][2]-1
vel = test['S'][0][0][6][:, idx]
idx = np.where((xc[:, 0]>230000) & (yc[:, 0] < -2500000))[0] # Spatial filtering
xc = xc[idx]
yc = yc[idx]

In [10]:
xext = [xc.min(), xc.max()]
yext = [yc.min(), yc.max()]
res = 200

gridx, gridy = np.meshgrid(np.arange(xext[0]+res, xext[1]-res, res), np.arange(yext[0]+res, yext[1]-res, res))
coord = (xc[:, 0], yc[:, 0]) #np.array([xc[:, 0], yc[:, 0]]).transpose()

print(gridx.shape)

(541, 425)


In [3]:
with open(f'D:\\ISSM\\Helheim\\valid_grid200_14297.pkl', 'rb') as file:
    grid_valid = pickle.load(file)

In [28]:
## Dataset for train ===================================

from scipy.interpolate import griddata

folder = "D:\\ISSM\\Helheim"
files = glob.glob(f"{folder}\\Helheim_*_030.mat")

for i, filename in enumerate(files[:]):
    
    dataset = GNN_Helheim_Dataset(files[i:i+1])
    
    print(filename)

    n_year = len(dataset) #, n_sample = H.shape
    n_samples = dataset[0].num_nodes()
    
    input0 = np.zeros((n_year, dataset[0].ndata['feat'].shape[1], gridx.shape[0], gridx.shape[1]))
    output0 = np.zeros((n_year, dataset[0].ndata['label'].shape[1], gridx.shape[0], gridx.shape[1])) 

    for t in tqdm(range(0, n_year)):
        
        inputs = dataset[t].ndata['feat']
        outputs = dataset[t].ndata['label']

        for c in range(0, inputs.shape[1]):
            input0[t, c, :, :] = griddata(coord, inputs[:, c], (gridx, gridy), method='nearest')
        for c in range(0, outputs.shape[1]):
            output0[t, c, :, :] = griddata(coord, outputs[:, c], (gridx, gridy), method='nearest')
    
    input0 = input0 * grid_valid
    output0 = output0 * grid_valid
    with open(filename.replace(".mat", f"_CNN_{int(res)}m.pkl"), 'wb') as file:
        pickle.dump([input0.astype(np.float16), output0.astype(np.float16)], file)

100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:21<00:00, 21.26s/it]


D:\ISSM\Helheim\Helheim_r065_030.mat


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:22<00:00, 22.26s/it]


D:\ISSM\Helheim\Helheim_r070_030.mat


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:23<00:00, 23.36s/it]


...

100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:26<00:00, 26.21s/it]


D:\ISSM\Helheim\Helheim_r115_030.mat


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:24<00:00, 24.63s/it]


D:\ISSM\Helheim\Helheim_r120_030.mat


100%|█████████████████████████████████████████████████████████████████████████████| 260/260 [18:20<00:00,  4.23s/it]
